# FraudMind Milestone 2 Demo

**Risk Aggregation node.** After all specialist scorers run, the graph passes through `risk_aggregation_node` which produces a `RiskVector`.

This notebook shows what `RiskVector` looks like across three cases:

| Case | Scenario | Tier | What to observe |
|---|---|---|---|
| 1 | Full ring attack: all 6 specialists with high signals | full_council (forced) | High aggregate, low stddev, ring dominant |
| 2 | Clean legitimate login: known device, home geo, no signals | standard | Near-zero aggregate, ring + ATO + payment all low |
| 3 | Ambiguous ATO only: new device + geo mismatch, nothing else | full_council | Low specialists_run, aggregate not enough to decide alone |

In [ ]:
import sys
sys.path.insert(0, '..')

from src.graph.fraud_graph import fraud_graph
from src.schemas.ato_schemas import MockATOSignals
from src.schemas.payment_schemas import PaymentSignals
from src.schemas.identity_schemas import IdentitySignals
from src.schemas.promo_schemas import PromoSignals
from src.schemas.ring_schemas import RingSignals
from src.schemas.payload_schemas import PayloadSignals
from src.config.scoring_config import RED_TEAM_VARIANCE_THRESHOLD

In [ ]:
def print_risk_vector(label: str, result: dict) -> None:
    """Print RiskVector fields from a graph invocation result."""
    rv = result.get('risk_vector')
    tier = result.get('routing_tier', 'unknown')

    print(f"{'=' * 60}")
    print(f"  {label}")
    print(f"  Routing tier : {tier}")
    print(f"{'=' * 60}")

    if rv is None:
        failures = result.get('failed_agents', [])
        print("  risk_vector  : not produced")
        if failures:
            print(f"  failures     : {', '.join(failures)}")
        print()
        return

    # Header metrics
    print(f"  aggregate    : {rv.aggregate:.4f}")
    print(f"  score_stddev : {rv.score_stddev:.4f}", end="")
    if rv.score_stddev >= RED_TEAM_VARIANCE_THRESHOLD:
        print("  <-- exceeds RED_TEAM_VARIANCE_THRESHOLD (specialists disagree)")
    else:
        print()
    print(f"  dominant     : {rv.dominant_domain}  ({rv.dominant_score:.4f})")
    print(f"  specialists  : {rv.specialists_run} ran  {rv.available_domains}")

    # Per-domain scores
    print()
    print("  Domain scores (highest to lowest):")
    for domain in rv.available_domains:
        bar_len = int(rv.scores[domain] * 30)
        bar = '#' * bar_len + '.' * (30 - bar_len)
        print(f"    {domain:<10} {rv.scores[domain]:.4f}  [{bar}]")

    # Specialist primary signals
    print()
    print("  Primary signals per specialist:")
    for key in ('ato_score', 'payment_score', 'identity_score',
                'promo_score', 'ring_score', 'payload_score'):
        s = result.get(key)
        if s is not None:
            domain = key.replace('_score', '')
            print(f"    {domain:<10} {', '.join(s.primary_signals)}")

    failures = result.get('failed_agents', [])
    if failures:
        print(f"\n  failures     : {', '.join(failures)}")

    print()

## Case 1: Full Ring Attack

All 6 specialists receive high-signal inputs. The ring has a confirmed signature match which forces `full_council` regardless of the caller-set tier.

**Expected:** High aggregate (> 0.70), ring is dominant, stddev likely low because most specialists agree this is high-risk.

In [ ]:
case1_state = {
    "primary_entity_id": "acc_ring_001",
    "primary_entity_type": "user",
    "event_type": "authentication",
    "routing_tier": "standard",  # will be forced to full_council by known_ring_signature_match

    "ato_signals": MockATOSignals(
        account_id="acc_ring_001",
        account_age_days=14,
        is_new_device=True,
        device_id="dev_shared_factory_01",
        login_geo="Lagos, Nigeria",
        account_home_geo="San Jose, CA",
        geo_mismatch=True,
        time_since_last_login_days=1,
        immediate_high_value_action=True,
        email_change_attempted=True,
        password_change_attempted=False,
        support_ticket_language_match=False,
    ),
    "payment_signals": PaymentSignals(
        transaction_id="txn_ring_001",
        account_id="acc_ring_001",
        amount_usd=1800.00,
        merchant_category="electronics",
        merchant_country="Nigeria",
        account_home_country="United States",
        is_international=True,
        transactions_last_1h=7,
        transactions_last_24h=12,
        avg_transaction_amount_30d=75.00,
        amount_vs_avg_ratio=24.0,
        is_first_transaction=False,
        card_present=False,
        days_since_account_creation=14,
    ),
    "identity_signals": IdentitySignals(
        account_id="acc_ring_001",
        account_age_days=14,
        email_domain="tempmail.io",
        is_disposable_email=True,
        phone_country_matches_ip_country=False,
        address_provided=True,
        address_is_commercial=True,
        document_verification_passed=False,
        pii_seen_on_other_accounts=6,
        accounts_created_from_same_device=22,
        accounts_created_from_same_ip=15,
        signup_velocity_same_ip_24h=8,
        name_dob_mismatch=True,
    ),
    "promo_signals": PromoSignals(
        account_id="acc_ring_001",
        promo_code="REF-RING-001",
        promo_type="referral",
        account_age_days=14,
        is_first_order=True,
        promo_uses_this_account=3,
        promo_uses_same_code=1,
        accounts_linked_same_device=10,
        accounts_linked_same_ip=12,
        referrer_account_id="acc_ring_ref_001",
        referrer_account_age_days=7,
        device_shared_with_referrer=True,
        email_domain_pattern_suspicious=True,
        order_cancelled_after_promo=True,
        payout_requested_immediately=True,
    ),
    "ring_signals": RingSignals(
        primary_entity_id="acc_ring_001",
        primary_entity_type="user",
        device_id="dev_shared_factory_01",
        ip_address_hash="hash_known_ring_ip_001",
        accounts_sharing_device=22,
        accounts_sharing_ip=15,
        linked_account_ids=["acc_ring_002", "acc_ring_003", "acc_ring_004"],
        linked_accounts_with_block_verdict=3,
        linked_accounts_with_fraud_confirmed=2,
        shared_payment_method_across_accounts=True,
        payment_method_account_count=7,
        known_ring_signature_match=True,
        ring_signature_id="ring_WEST_AFRICA_001",
        velocity_account_creation_same_ip_7d=18,
        transaction_graph_min_hops_to_fraud=1,
    ),
    "payload_signals": PayloadSignals(
        session_id="sess_ring_001",
        account_id="acc_ring_001",
        user_agent_string="HeadlessChrome/118.0",
        user_agent_is_headless=True,
        tls_fingerprint_ja3="a1b2c3d4e5f6a1b2c3d4e5f6a1b2c3d4",
        tls_fingerprint_known_bot=True,
        http_header_order_anomaly=True,
        accept_language_missing=True,
        mouse_movement_entropy=0.0,
        keystroke_dynamics_score=0.04,
        requests_per_minute=210.0,
        request_timing_variance_ms=2.1,
        captcha_solve_time_ms=85,
        captcha_solve_pattern_automated=True,
        credential_stuffing_ip_on_blocklist=True,
        login_attempt_count_this_session=9,
        api_endpoint_sequence_anomaly=True,
    ),
}

result1 = fraud_graph.invoke(case1_state)
print_risk_vector("Case 1: Full Ring Attack (forced full_council)", result1)

## Case 2: Clean Legitimate Login

Known device used 47 times before, login from account home geo, no suspicious actions. Standard tier: ATO + Payment + Ring only.

**Expected:** Near-zero aggregate, no dominant risk domain, stddev near zero (all specialists agree this is clean).

In [ ]:
case2_state = {
    "primary_entity_id": "acc_clean_002",
    "primary_entity_type": "user",
    "event_type": "authentication",
    "routing_tier": "standard",

    "ato_signals": MockATOSignals(
        account_id="acc_clean_002",
        account_age_days=847,
        is_new_device=False,
        device_id="dev_trusted_047",
        login_geo="San Jose, CA",
        account_home_geo="San Jose, CA",
        geo_mismatch=False,
        time_since_last_login_days=1,
        immediate_high_value_action=False,
        email_change_attempted=False,
        password_change_attempted=False,
        support_ticket_language_match=True,
    ),
    "payment_signals": PaymentSignals(
        transaction_id="txn_clean_002",
        account_id="acc_clean_002",
        amount_usd=94.50,
        merchant_category="grocery",
        merchant_country="United States",
        account_home_country="United States",
        is_international=False,
        transactions_last_1h=1,
        transactions_last_24h=3,
        avg_transaction_amount_30d=88.00,
        amount_vs_avg_ratio=1.07,
        is_first_transaction=False,
        card_present=True,
        days_since_account_creation=847,
    ),
    "ring_signals": RingSignals(
        primary_entity_id="acc_clean_002",
        primary_entity_type="user",
        device_id="dev_trusted_047",
        ip_address_hash="hash_home_ip_002",
        accounts_sharing_device=1,
        accounts_sharing_ip=1,
        linked_account_ids=[],
        linked_accounts_with_block_verdict=0,
        linked_accounts_with_fraud_confirmed=0,
        shared_payment_method_across_accounts=False,
        payment_method_account_count=1,
        known_ring_signature_match=False,
        ring_signature_id=None,
        velocity_account_creation_same_ip_7d=1,
        transaction_graph_min_hops_to_fraud=99,
    ),
}

result2 = fraud_graph.invoke(case2_state)
print_risk_vector("Case 2: Clean Legitimate Login (standard tier)", result2)

## Case 3: Ambiguous ATO Only

New device, geo mismatch, but no payment, ring, identity, promo, or payload signals available. Only the ATO specialist runs.

**Expected:** `specialists_run = 1`, aggregate reflects ATO score only, `available_domains = ['ato']`.

This illustrates a key limitation: with a single specialist the aggregate equals that domain's raw score and `score_stddev = 0.0`. There is no cross-domain corroboration to raise or lower confidence. The Arbiter (Milestone 3) will use `specialists_run` and `score_stddev` alongside `aggregate` to decide whether the evidence base is sufficient or whether the case needs escalation for more signal collection.

In [ ]:
case3_state = {
    "primary_entity_id": "acc_ambig_003",
    "primary_entity_type": "user",
    "event_type": "authentication",
    "routing_tier": "full_council",  # full_council requested but only ATO signals available

    "ato_signals": MockATOSignals(
        account_id="acc_ambig_003",
        account_age_days=210,
        is_new_device=True,
        device_id="dev_unknown_003",
        login_geo="London, UK",
        account_home_geo="New York, NY",
        geo_mismatch=True,
        time_since_last_login_days=12,
        immediate_high_value_action=False,
        email_change_attempted=False,
        password_change_attempted=False,
        support_ticket_language_match=True,
    ),
    # No payment_signals, ring_signals, identity_signals, promo_signals, payload_signals
}

result3 = fraud_graph.invoke(case3_state)
print_risk_vector("Case 3: Ambiguous ATO Only (1 specialist ran)", result3)

rv3 = result3.get('risk_vector')
if rv3:
    print("Interpretation:")
    print(f"  specialists_run = {rv3.specialists_run} means there is no cross-domain corroboration.")
    print(f"  score_stddev    = {rv3.score_stddev:.4f} (zero because only one domain contributed)")
    print(f"  aggregate       = {rv3.aggregate:.4f} -- reflects ATO score only, not a council consensus.")
    print()
    print("  The Arbiter (Milestone 3) will factor specialists_run into its verdict logic.")
    print("  A single-specialist aggregate above the block threshold should produce 'escalate'")
    print("  rather than 'block', since no other domain has confirmed or denied the risk.")